<a href="https://colab.research.google.com/github/qiyuriko/ICT304-Assignment1/blob/main/ICT304_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
!unzip -q "/content/bird dataset.zip" -d /content/bird_data
print("Unzip completed. Contents extracted to /content/bird_data")

Unzip completed. Contents extracted to /content/bird_data


In [5]:
import os
from collections import Counter

# The dataset unzips into "/content/bird_data/bird dataset"
base_path = "/content/bird_data/bird dataset"

if os.path.exists(base_path):
    print(f"--- Exploring Dataset Classes in {base_path} ---")
    for folder in os.listdir(base_path):
        folder_path = os.path.join(base_path, folder)

        if os.path.isdir(folder_path):
            files = os.listdir(folder_path)
            extensions = Counter(os.path.splitext(f)[1].lower() for f in files)

            print(f"\nClass: {folder}")
            print("Total files:", len(files))
            print("Formats:", extensions)
else:
    print(f"Directory not found: {base_path}. Please make sure the dataset is extracted successfully.")

--- Exploring Dataset Classes in /content/bird_data/bird dataset ---

Class: marsh sandpiper
Total files: 140
Formats: Counter({'.jpg': 140})

Class: hooded merganser
Total files: 60
Formats: Counter({'.jpg': 60})

Class: common kingfisher
Total files: 85
Formats: Counter({'.jpg': 79, '.png': 6})

Class: greater cormorant
Total files: 140
Formats: Counter({'.jpg': 140})

Class: mallard
Total files: 60
Formats: Counter({'.jpg': 60})

Class: black crowned night heron
Total files: 156
Formats: Counter({'.jpg': 140, '.png': 16})


In [6]:
import os

def list_dir_structure(startpath, max_depth=3):
    for root, dirs, files in os.walk(startpath):
        depth = root.replace(startpath, '').count(os.sep)
        if depth < max_depth:
            indent = ' ' * 4 * (depth)
            print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')

list_dir_structure('/content/bird_data')

bird_data/ (0 files)
    bird dataset/ (0 files)
        marsh sandpiper/ (140 files)
        hooded merganser/ (60 files)
        common kingfisher/ (85 files)
        greater cormorant/ (140 files)
        mallard/ (60 files)
        black crowned night heron/ (156 files)


In [7]:
import shutil
import os

# Remove existing directory if it exists
bird_data_path = '/content/bird_data'
if os.path.exists(bird_data_path):
    shutil.rmtree(bird_data_path)
    print(f"Removed {bird_data_path} and all its contents.")

# Recreate clean folder
os.makedirs(bird_data_path, exist_ok=True)

# Unzip the uploaded dataset into it
!unzip -q "/content/bird dataset.zip" -d /content/bird_data
print("Re-extraction completed successfully!")

Removed /content/bird_data and all its contents.
Re-extraction completed successfully!


In [8]:
# Verify the cleaned and newly extracted folder structure
list_dir_structure('/content/bird_data')

bird_data/ (0 files)
    bird dataset/ (0 files)
        marsh sandpiper/ (140 files)
        hooded merganser/ (60 files)
        common kingfisher/ (85 files)
        greater cormorant/ (140 files)
        mallard/ (60 files)
        black crowned night heron/ (156 files)


In [9]:
import os
from PIL import Image
from collections import Counter

input_base = "/content/bird_data/bird dataset"
output_base = "/content/bird_data/preprocessed/color"
target_size = 224

def to_square(img, size=224, resample=Image.BICUBIC):
    w, h = img.size
    scale = size / max(w, h)
    img = img.resize((round(w * scale), round(h * scale)), resample)
    canvas = Image.new("RGB", (size, size), (0, 0, 0))
    canvas.paste(img, ((size - img.width) // 2, (size - img.height) // 2))
    return canvas

# Preprocess the 6 species
for bird_class in os.listdir(input_base):
    input_folder = os.path.join(input_base, bird_class)
    if not os.path.isdir(input_folder):
        continue

    output_folder = os.path.join(output_base, bird_class)
    os.makedirs(output_folder, exist_ok=True)

    for file in os.listdir(input_folder):
        input_file = os.path.join(input_folder, file)
        output_file = os.path.join(output_folder, file)

        try:
            with Image.open(input_file) as img:
                img = img.convert("RGB")
                img = to_square(img, target_size)
                img.save(output_file)
        except Exception as e:
            print(f"Error processing {input_file}: {e}")

print("Preprocessing completed successfully for 6 species!")

Preprocessing completed successfully for 6 species!


In [10]:
# Explore preprocessed classes and counts
print("Preprocessed Class Counts:")
for c in sorted(os.listdir(output_base)):
    path = os.path.join(output_base, c)
    if os.path.isdir(path):
        print(f"{c}: {len(os.listdir(path))} images")

Preprocessed Class Counts:
black crowned night heron: 156 images
common kingfisher: 85 images
greater cormorant: 140 images
hooded merganser: 60 images
mallard: 60 images
marsh sandpiper: 140 images


In [11]:
# Check for corrupted images in the preprocessed folder
from PIL import Image
import os

base_path = "/content/bird_data/preprocessed/color"
corrupted = []

for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file)
            try:
                with Image.open(file_path) as img:
                    img.verify()
            except Exception:
                corrupted.append(file_path)

print(f"Total corrupted images found: {len(corrupted)}")

Total corrupted images found: 0


In [12]:
# Explore preprocessed image sizes and color modes
from collections import Counter

base_path = "/content/bird_data/preprocessed/color"
sizes = []
modes = []

for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file)
            with Image.open(file_path) as img:
                sizes.append(img.size)
                modes.append(img.mode)

print("Image sizes:", Counter(sizes))
print("Color modes:", Counter(modes))

Image sizes: Counter({(224, 224): 641})
Color modes: Counter({'RGB': 641})


In [18]:
import os
import shutil
import random
import numpy as np
from sklearn.model_selection import StratifiedKFold

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

preprocessed_base = "/content/bird_data/preprocessed/color"
kfold_base_dir = "/content/bird_data/kfold_split"
test_dir = "/content/bird_data/kfold_split/test"

# Clean up any existing split directories to avoid mixing
if os.path.exists(kfold_base_dir):
    shutil.rmtree(kfold_base_dir)
os.makedirs(test_dir, exist_ok=True)

# Collect all file paths and their respective labels
all_filepaths = []
all_labels = []

bird_classes = sorted([d for d in os.listdir(preprocessed_base) if os.path.isdir(os.path.join(preprocessed_base, d))])

for label_idx, bird_class in enumerate(bird_classes):
    class_path = os.path.join(preprocessed_base, bird_class)
    images = sorted(os.listdir(class_path))
    for img in images:
        all_filepaths.append(os.path.join(class_path, img))
        all_labels.append(label_idx)

all_filepaths = np.array(all_filepaths)
all_labels = np.array(all_labels)

# Step 1: Separate a clean, stratified 10% hold-out test set
skf_test = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
train_val_indices, test_indices = next(skf_test.split(all_filepaths, all_labels))

train_val_paths, train_val_labels = all_filepaths[train_val_indices], all_labels[train_val_indices]
test_paths, test_labels = all_filepaths[test_indices], all_labels[test_indices]

# Copy hold-out test set to its directory
for path in test_paths:
    bird_class = os.path.basename(os.path.dirname(path))
    dest_class_dir = os.path.join(test_dir, bird_class)
    os.makedirs(dest_class_dir, exist_ok=True)
    shutil.copy(path, os.path.join(dest_class_dir, os.path.basename(path)))

# Step 2: Split the remaining 90% into 5 Stratified Folds
skf_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("--- Starting Stratified 5-Fold Splitting ---")
for fold, (train_idx, val_idx) in enumerate(skf_cv.split(train_val_paths, train_val_labels)):
    fold_dir = os.path.join(kfold_base_dir, f"fold_{fold}")
    fold_train_dir = os.path.join(fold_dir, "train")
    fold_val_dir = os.path.join(fold_dir, "val")

    # Copy training set for this fold
    for idx in train_idx:
        path = train_val_paths[idx]
        bird_class = os.path.basename(os.path.dirname(path))
        dest_dir = os.path.join(fold_train_dir, bird_class)
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copy(path, os.path.join(dest_dir, os.path.basename(path)))

    # Copy validation set for this fold
    for idx in val_idx:
        path = train_val_paths[idx]
        bird_class = os.path.basename(os.path.dirname(path))
        dest_dir = os.path.join(fold_val_dir, bird_class)
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copy(path, os.path.join(dest_dir, os.path.basename(path)))

    print(f"Fold {fold} generated successfully with {len(train_idx)} train images and {len(val_idx)} validation images.")

print("\nAll folds and hold-out test set generated successfully!")

--- Starting Stratified 5-Fold Splitting ---
Fold 0 generated successfully with 460 train images and 116 validation images.
Fold 1 generated successfully with 461 train images and 115 validation images.
Fold 2 generated successfully with 461 train images and 115 validation images.
Fold 3 generated successfully with 461 train images and 115 validation images.
Fold 4 generated successfully with 461 train images and 115 validation images.

All folds and hold-out test set generated successfully!


In [19]:
# Verify the 5-Fold split distribution structure
print("--- K-Fold Verification ---")
for fold in range(5):
    fold_dir = os.path.join(kfold_base_dir, f"fold_{fold}")
    train_count = sum(len(files) for r, d, files in os.walk(os.path.join(fold_dir, "train")))
    val_count = sum(len(files) for r, d, files in os.walk(os.path.join(fold_dir, "val")))
    print(f"Fold {fold}: Train size = {train_count} images | Val size = {val_count} images")

test_count = sum(len(files) for r, d, files in os.walk(test_dir))
print(f"\nHold-out Test Set size: {test_count} images")

--- K-Fold Verification ---
Fold 0: Train size = 460 images | Val size = 116 images
Fold 1: Train size = 461 images | Val size = 115 images
Fold 2: Train size = 461 images | Val size = 115 images
Fold 3: Train size = 461 images | Val size = 115 images
Fold 4: Train size = 461 images | Val size = 115 images

Hold-out Test Set size: 65 images


In [20]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define data augmentation for training
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Define basic transforms for validation & test (no augmentation, only normalization)
val_test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def get_kfold_loaders(fold, batch_size=32):
    """Helper function to fetch PyTorch DataLoaders for a specific fold."""
    fold_dir = f"/content/bird_data/kfold_split/fold_{fold}"
    train_dataset = datasets.ImageFolder(root=os.path.join(fold_dir, "train"), transform=train_transforms)
    val_dataset = datasets.ImageFolder(root=os.path.join(fold_dir, "val"), transform=val_test_transforms)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader

# Create a separate Loader for the hold-out test set
test_dataset = datasets.ImageFolder(root="/content/bird_data/kfold_split/test", transform=val_test_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print("Data Augmentation and K-Fold Loaders initialized successfully!")

Data Augmentation and K-Fold Loaders initialized successfully!
